Coding Challenge: 8-Puzzle Solving Agent
---


**Dr Chao Shu (chao.shu@qmul.ac.uk)**


Submitted By

**Name**:

**QMUL ID**:

**BUPT ID**:


## Introduction

In this coding challenge, you are given only the 8-puzzle environment and evaluation requirements.
Your task is to design an LLM-based agent with tool calling that can solve 8-puzzle instances and satisfy strict grading constraints.

You are free to choose the method(s) you learned in this module, but your final system must satisfy all requirements below.

> ‼️ This is an individual task. You are only allowed to work with your AI assistant(s). Help from classmates is NOT allowed.
>
> ‼️ Signs used in tasks:
>
>    💬: The TAs/lecturer may ask follow-up questions to test your understanding.
>
>    🧑‍💻: You need to run and demonstrate the result to the TAs/lecturer.


## Requirements and Grading Constraints

### Core requirement
Build an LLM tool-calling agent that returns **valid** solutions and achieves **optimal** solution length on evaluation puzzles.

### Efficiency constraints (per puzzle)
- Maximum `4` LLM turns
- Maximum `6` tool calls

### Interface contract (must implement)
`run_agent_on_puzzle(initial_state, student_id) -> dict`

The returned dict must contain:
- `actions` (`list[str]`)
- `llm_turns` (`int`)
- `tool_calls` (`int`)
- `used_final_answer` (`bool`)

### Demonstration requirement
Use the provided `show_react_step()` helper to display each ReAct step (assistant output + observation).


In [ ]:
import hashlib
import json
import random
from typing import Any

import numpy as np
from dotenv import load_dotenv

from utils import get_completion, print_in_box, LLMModels

load_dotenv()

MAX_LLM_TURNS = 4
MAX_TOOL_CALLS = 6
PUBLIC_SALT_V1 = "PUBLIC_SALT_V1"

# Configure the model endpoint used by the agent.
# MODEL_API_CONFIG = LLMModels.OLLAMA_QWEN_2_5_1_5B.value
MODEL_API_CONFIG = LLMModels.GITHUB_GPT_41_MINI.value

print(f"Model config: {MODEL_API_CONFIG.provider}/{MODEL_API_CONFIG.name}")


## Step 1: 8-Puzzle Environment

Use this class as the environment interface.
Your agent/tooling must interact with this state representation.


In [ ]:
class SlidingPuzzleState:
    def __init__(self, state=None, size=3):
        self.size = size

        if state is None:
            self.state = self._generate_goal_state()
        else:
            self.state = np.array(state)
            if self.state.shape != (size, size):
                raise ValueError(f"Provided state must be a {size}x{size} grid")

        blank_pos = np.where(self.state == 0)
        self.blank_row, self.blank_col = blank_pos[0][0], blank_pos[1][0]
        self.goal_state = self._generate_goal_state()

    def _generate_goal_state(self):
        goal = np.arange(1, self.size * self.size + 1).reshape(self.size, self.size)
        goal[-1, -1] = 0
        return goal

    def is_goal(self):
        return np.array_equal(self.state, self.goal_state)

    def get_possible_actions(self):
        actions = []
        if self.blank_row > 0:
            actions.append("up")
        if self.blank_row < self.size - 1:
            actions.append("down")
        if self.blank_col > 0:
            actions.append("left")
        if self.blank_col < self.size - 1:
            actions.append("right")
        return actions

    def apply_action(self, action):
        new_state = np.copy(self.state)

        if action == "up":
            new_state[self.blank_row][self.blank_col] = new_state[self.blank_row - 1][self.blank_col]
            new_state[self.blank_row - 1][self.blank_col] = 0
        elif action == "down":
            new_state[self.blank_row][self.blank_col] = new_state[self.blank_row + 1][self.blank_col]
            new_state[self.blank_row + 1][self.blank_col] = 0
        elif action == "left":
            new_state[self.blank_row][self.blank_col] = new_state[self.blank_row][self.blank_col - 1]
            new_state[self.blank_row][self.blank_col - 1] = 0
        elif action == "right":
            new_state[self.blank_row][self.blank_col] = new_state[self.blank_row][self.blank_col + 1]
            new_state[self.blank_row][self.blank_col + 1] = 0
        else:
            raise ValueError(f"Invalid action: {action}")

        return SlidingPuzzleState(new_state, self.size)

    def display(self):
        max_num = self.size * self.size - 1
        spacing = len(str(max_num)) + 1
        for row in self.state:
            row_str = ""
            for val in row:
                row_str += f"{val:>{spacing-1}} "
            print(row_str)

    def to_list(self):
        return self.state.tolist()

    def __eq__(self, other):
        return isinstance(other, SlidingPuzzleState) and np.array_equal(self.state, other.state)

    def __hash__(self):
        return hash(tuple(self.state.flatten()))

    def __lt__(self, other):
        return tuple(self.state.flatten()) < tuple(other.state.flatten())


## Step 2: Public Puzzles

Use your **10-digit student ID** to generate personalised public puzzle instances.


In [ ]:
def validate_student_id(student_id: int) -> int:
    if not isinstance(student_id, int):
        raise TypeError("student_id must be an integer")
    if student_id < 1_000_000_000 or student_id > 9_999_999_999:
        raise ValueError("student_id must be a 10-digit integer")
    return student_id


def seed_from_id(student_id: int, salt: str) -> int:
    token = f"{student_id}:{salt}".encode("utf-8")
    digest = hashlib.sha256(token).hexdigest()
    return int(digest[:16], 16)


def scramble_from_goal(seed: int, depth: int, size: int = 3) -> list[list[int]]:
    rng = random.Random(seed)
    state = SlidingPuzzleState(size=size)

    reverse_action = {
        "up": "down",
        "down": "up",
        "left": "right",
        "right": "left",
    }

    prev_action = None
    for _ in range(depth):
        actions = state.get_possible_actions()
        if prev_action is not None and reverse_action[prev_action] in actions and len(actions) > 1:
            actions.remove(reverse_action[prev_action])
        action = rng.choice(actions)
        state = state.apply_action(action)
        prev_action = action

    return state.to_list()


def generate_public_puzzles(student_id: int, num_puzzles: int = 1) -> list[dict[str, Any]]:
    student_id = validate_student_id(student_id)
    base_seed = seed_from_id(student_id, PUBLIC_SALT_V1)

    # Increasing scramble depths for varied difficulty.
    depths = [8, 12, 16, 20, 24][:num_puzzles]

    puzzles = []
    for idx, depth in enumerate(depths, start=1):
        puzzle_seed = base_seed + idx * 10_003
        initial_state = scramble_from_goal(seed=puzzle_seed, depth=depth, size=3)
        puzzles.append(
            {
                "puzzle_id": f"public_{idx}",
                "initial_state": initial_state,
                "scramble_depth": depth,
            }
        )

    return puzzles


In [ ]:
# TODO: Set your 10-digit student ID
student_id = 2023123456

# Default local test uses one personalized public puzzle.
public_puzzles = generate_public_puzzles(student_id, num_puzzles=5)
print(f"Generated {len(public_puzzles)} public puzzle(s) for student_id={student_id}")

for puzzle in public_puzzles:
    print()
    print(puzzle["puzzle_id"], "| scramble_depth=", puzzle["scramble_depth"])
    SlidingPuzzleState(puzzle["initial_state"]).display()


## Step 3: Mandatory Step-by-Step Demonstration Output

Do not write log files or structured process logs.

Instead, during agent execution, use `show_react_step()` to display each step clearly:
- assistant response (THINK/ACT)
- observation/tool result
- final answer

Your agent must finish by calling the `final_answer(...)` tool.

The `final_answer(...)` tool function is provided below; include it in your toolset and use it for the final response.


In [ ]:
def show_react_step(step_idx: int, assistant_text: str, observation_text: str):
    print_in_box(assistant_text, title=f"Assistant Step {step_idx}")
    print_in_box(observation_text, title=f"Observe Step {step_idx}")


def final_answer(answer: str) -> dict[str, str]:
    """Provided final-answer tool. Students should include and use this tool."""
    return {"answer": answer}


## Step 4: Implement the Agent Interface

Implement the required function below.

`run_agent_on_puzzle(initial_state, student_id) -> dict`

Return dict keys (all required):
- `actions`: list of moves
- `llm_turns`: number of LLM rounds used
- `tool_calls`: number of tool calls used
- `used_final_answer`: `True` only if the agent called `final_answer(...)`

Also show the step-by-step reasoning/tool interaction using `show_react_step()`.


In [ ]:
# TODO: Implement your LLM tool-calling agent.
# Notes:
# - You may reuse/adapt the ReAct orchestration style from L06.
# - Your method choice is open, but hidden grading checks optimality and budgets.
# - You must call the provided final_answer(...) tool and set used_final_answer=True when that happens.
# - Use show_react_step() to display each assistant step and each observation.


def run_agent_on_puzzle(initial_state, student_id) -> dict:
    """
    Required output schema:
    {
        "actions": list[str],
        "llm_turns": int,
        "tool_calls": int,
        "used_final_answer": bool,
    }
    """
    student_id = validate_student_id(int(student_id))

    # TODO: replace placeholders with your implementation.
    show_react_step(
        step_idx=1,
        assistant_text="THINK:\nTODO\nACT:\nsolve_with_search(...)",
        observation_text="OBSERVE:\n{'actions': ['up', 'left']}",
    )

    return {
        "actions": [],
        "llm_turns": 0,
        "tool_calls": 0,
        "used_final_answer": False,
    }


## 🧑‍💻 Step 5: Public Local Evaluator

This evaluator checks interface, validity, and budget compliance on your personalized public set.

In [ ]:
def validate_action_sequence(initial_state: list[list[int]], actions: list[str]) -> tuple[bool, str]:
    current = SlidingPuzzleState(initial_state)

    for step, action in enumerate(actions, start=1):
        valid_actions = current.get_possible_actions()
        if action not in valid_actions:
            return False, f"Invalid action `{action}` at step {step}; valid actions were {valid_actions}."
        current = current.apply_action(action)

    if current.is_goal():
        return True, "Reached goal state."
    return False, "Action sequence ended before reaching goal state."


def evaluate_submission_on_public_set(run_fn, public_puzzles: list[dict[str, Any]], student_id: int):
    summary = []

    for puzzle in public_puzzles:
        puzzle_id = puzzle["puzzle_id"]
        initial_state = puzzle["initial_state"]

        try:
            result = run_fn(initial_state, student_id)
        except Exception as exc:
            summary.append(
                {
                    "puzzle_id": puzzle_id,
                    "passed": False,
                    "reason": f"runtime error: {type(exc).__name__}: {exc}",
                }
            )
            continue

        required_keys = {"actions", "llm_turns", "tool_calls", "used_final_answer"}
        if not isinstance(result, dict) or set(result.keys()) != required_keys:
            summary.append(
                {
                    "puzzle_id": puzzle_id,
                    "passed": False,
                    "reason": "result schema mismatch",
                }
            )
            continue

        if not isinstance(result["actions"], list):
            summary.append({"puzzle_id": puzzle_id, "passed": False, "reason": "actions must be a list"})
            continue

        if not isinstance(result["used_final_answer"], bool):
            summary.append({"puzzle_id": puzzle_id, "passed": False, "reason": "used_final_answer must be a bool"})
            continue

        if not result["used_final_answer"]:
            summary.append({"puzzle_id": puzzle_id, "passed": False, "reason": "final_answer tool was not used"})
            continue

        if result["llm_turns"] > MAX_LLM_TURNS:
            summary.append(
                {
                    "puzzle_id": puzzle_id,
                    "passed": False,
                    "reason": f"llm_turns exceeded budget ({result['llm_turns']} > {MAX_LLM_TURNS})",
                }
            )
            continue

        if result["tool_calls"] > MAX_TOOL_CALLS:
            summary.append(
                {
                    "puzzle_id": puzzle_id,
                    "passed": False,
                    "reason": f"tool_calls exceeded budget ({result['tool_calls']} > {MAX_TOOL_CALLS})",
                }
            )
            continue

        valid, message = validate_action_sequence(initial_state, result["actions"])
        if not valid:
            summary.append({"puzzle_id": puzzle_id, "passed": False, "reason": message})
            continue

        summary.append(
            {
                "puzzle_id": puzzle_id,
                "passed": True,
                "reason": "valid solution + final_answer + budget checks passed",
                "llm_turns": result["llm_turns"],
                "tool_calls": result["tool_calls"],
                "solution_length": len(result["actions"]),
            }
        )

    passed = sum(1 for s in summary if s.get("passed"))
    print(f"Public evaluation: {passed}/{len(summary)} puzzles passed")
    for row in summary:
        print(row)

    return summary


In [ ]:
# TODO: Run your local public evaluation after implementing run_agent_on_puzzle.

# public_results = evaluate_submission_on_public_set(
#     run_fn=run_agent_on_puzzle,
#     public_puzzles=public_puzzles,
#     student_id=student_id,
# )


## Reflection Questions

💬 **Q1:** Explain your agent architecture and why it can satisfy both optimality and interaction budgets.


*Your answer here*

💬 **Q2:** How can you make sure to achieve the optimal solution (fewest steps) of the 8-puzzle problem?


*Your answer here*

💬 **Q3:** What GenAI coding tool did you use?


*Your answer here*

## Submission Checklist

- [ ] Implemented `run_agent_on_puzzle(initial_state, student_id) -> dict` with required keys
- [ ] Works on personalized public puzzle generated from your 10-digit `student_id`
- [ ] Meets per-puzzle budgets (`llm_turns <= 4`, `tool_calls <= 6`)
- [ ] Uses `final_answer(...)` for the final answer (`used_final_answer=True`)
- [ ] Uses `show_react_step()` to clearly demonstrate each assistant/tool step
- [ ] Answered all reflection questions
- [ ] Included conversation thread(s) with your AI assistant(s)
